# E1: Softmax, MLP and CNN on CIFAR-10 (replication notebook)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/broistg/deep-learning-CO5085/blob/main/E1/notebooks/E1.ipynb)

Thin notebook: all code lives in `E1/e1lib/` and `common/`. Two modes (set `MODE` in the next cell):

| `MODE` | What it does | Runtime | Time |
|---|---|---|---|
| `"load"` (default) | Regenerates the summary table and 3-seed learning curves from committed results; confusion matrices and error galleries show seed 0. Writes the table to E1/results and figures to E1/figures. | free Colab; CPU is enough | a few minutes (CIFAR-10 download included) |
| `"rerun"` | Retrains every model (Softmax, MLP, MiniVGG x seeds 0,1,2, 30 epochs, plus the no-augmentation runs if they are in the plan) into a fresh `E1/results/`; the published results are kept in `E1/results_published/` and compared | GPU runtime | the notebook prints an estimate from the published timings first |

A rerun is not bit-identical (another GPU), so it is judged by the mean per configuration against the published standard deviation. Protocol and changes: `E1/README.md`. The original single-notebook version is in `E1/notebooks/archive/E1_original.ipynb`.

In [ ]:
# 0. Settings
MODE = "load"            # "load" (minutes, CPU ok) or "rerun" (GPU)
REPO_URL = "https://github.com/broistg/deep-learning-CO5085.git"
BRANCH = "main"

In [ ]:
# 1. Code: clone or update the repo, install the dependencies (torch is NOT reinstalled), put the repo on the path
import os, sys
ON_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/repo" if ON_COLAB else os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ON_COLAB and not os.path.isdir(REPO_DIR + "/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
if ON_COLAB:
    !git pull --ff-only
!pip install -q -r requirements/e.txt
!pip install -q -e .
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from common import notebook_utils as nu

In [ ]:
# 2. What this run will do
nu.banner(MODE, "E1")
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if MODE == "rerun":
    assert torch.cuda.is_available(), "rerun needs a GPU runtime (Runtime > Change runtime type)"
    h, n = nu.estimate_hours("E1/results")
    print(f"Estimate from the published runs: {n} runs, about {h:.1f} h of pure training time on the original machine (plus evaluation overhead).")

In [ ]:
# 3. Tests, then CIFAR-10 and the frozen split (the hash must be the same on every machine)
!python -m unittest discover -s E1/tests -v
!python -m common.cifar

## Load mode

In [ ]:
# 4a. (load) regenerate the summary table and the figures from saved runs into E1/results and E1/figures
if MODE == "load":
    !python -m e1lib.summary
    !python -m e1lib.analysis --seed 0 --out E1/figures

## Rerun mode

In [ ]:
# 5a. (rerun) dry run on synthetic data, then move the published results aside
if MODE == "rerun":
    !python -m e1lib.run --model mlp --seed 0 --fake --smoke
    nu.stash_published("E1")

In [ ]:
# 5b. (rerun) all runs. Resumable: if Colab disconnects, run the notebook again (MODE="rerun") and finished runs are skipped.
if MODE == "rerun":
    !python -m e1lib.run --all

In [ ]:
# 5c. (rerun) compare with the published results, then regenerate summary and figures from the new runs
if MODE == "rerun":
    nu.compare_means("E1/results", "E1/results_published")
    nu.compare_runs("E1/results", "E1/results_published")
    !python -m e1lib.summary
    !python -m e1lib.analysis --seed 0 --out E1/figures

## Figures and table

In [ ]:
# 6. Show the figures and the table
from pathlib import Path
nu.show_pngs("E1/figures")
print(Path("E1/results/tab_e1_summary.tex").read_text(encoding="utf8"))

Load mode updates the summary/table in E1/results and figures in E1/figures. To restore published runs after a rerun: `nu.restore_published("E1")`.